In [10]:
import math

from PIL import Image

8.      Реализовать 3D геометрические преобразования для «проволочного» рендера.

In [13]:
def Brezenheim_otrezok(img, x0, y0, x1, y1):
    b = int((img.size[0] - 1) / 2)
    pixels = img.load()
    steep = abs(y1 - y0) > abs(x1 - x0)
    if steep:
        x0, y0 = y0, x0
        x1, y1 = y1, x1
    if x0 > x1:
        x0, x1 = x1, x0
        y0, y1 = y1, y0
    dx = x1 - x0
    dy = abs(y1 - y0)
    d = 2 * dy - dx
    y = y0
    step_y = 1 if y1 > y0 else -1
    for x in range(x0, x1 + 1):
        curr_x, curr_y = (y, x) if steep else (x, y)
        if 0 <= b + curr_x < img.size[0] and 0 <= b - curr_y < img.size[1]:
            pixels[b + curr_x, b - curr_y] = 0

        if d > 0:
            y += step_y
            d -= 2 * dx
        d += 2 * dy


def transform_scale(vertices, sx, sy, sz):
    return [[x * sx, y * sy, z * sz] for x, y, z in vertices]


def transform_translate(vertices, tx, ty, tz):
    return [[x + tx, y + ty, z + tz] for x, y, z in vertices]


def transform_rotate_z(vertices, angle_rad):
    c = math.cos(angle_rad)
    s = math.sin(angle_rad)
    return [[x * c - y * s, x * s + y * c, z] for x, y, z in vertices]


def transform_twist_x(vertices, alpha):
    new_vertices = []
    for x, y, z in vertices:
        c = math.cos(alpha * x)
        s = math.sin(alpha * x)
        new_y = y * c - z * s
        new_z = y * s + z * c
        new_vertices.append([x, new_y, new_z])
    return new_vertices


def transform_bend_z(vertices, gamma):
    new_vertices = []
    for x, y, z in vertices:
        r = math.sqrt(x ** 2 + y ** 2)
        c = math.cos(gamma * r)
        s = math.sin(gamma * r)

        new_x = x * c - y * s
        new_y = x * s + y * c
        new_vertices.append([new_x, new_y, z])
    return new_vertices


def render(file, transform_type='none', param=0.0):
    vertices = []
    faces = []

    with open(file, 'r') as f:
        for line in f:
            if line.startswith('v '):
                vertices.append(list(map(float, line.split()[1:4])))
            elif line.startswith('f '):
                faces.append([int(i.split('/')[0]) - 1 for i in line.split()[1:]])

    if transform_type == 'scale':
        vertices = transform_scale(vertices, param[0], param[1], param[2])
    elif transform_type == 'translate':
        vertices = transform_translate(vertices, param[0], param[1], param[2])
    elif transform_type == 'rotate_z':
        vertices = transform_rotate_z(vertices, param)
    elif transform_type == 'twist':
        vertices = transform_twist_x(vertices, param)
    elif transform_type == 'bend':
        vertices = transform_bend_z(vertices, param)
    img = Image.new('L', (501, 501), color=255)

    def project(vertex):
        x = int(vertex[0] * 150)
        y = int(vertex[1] * 150)
        return x, y

    for face in faces:
        for i in range(len(face)):
            v1 = vertices[face[i]]
            v2 = vertices[face[(i + 1) % len(face)]]
            x0, y0 = project(v1)
            x1, y1 = project(v2)
            Brezenheim_otrezok(img, x0, y0, x1, y1)

    img.show()


obj_file = 'african_head.obj'
# 1. Обычный рендер без изменений
render(obj_file)
# 2. Масштабирование (увеличить в 1.5 раза по всем осям)
render(obj_file, transform_type='scale', param=(1.5, 1.5, 1.5))
# 3. Поворот вокруг оси Z на 45 градусов
render(obj_file, transform_type='rotate_z', param=math.radians(45))
# 4. Кручение (Twist)
render(obj_file, transform_type='twist', param=1)
# 5. Изгиб (Bend)
render(obj_file, transform_type='bend', param=1)